# RiCTO runner (`c_Run_Tools/optimization`)

- 부호·솔버 검증: `validate_ricto.py --dummy --solvers`
- synthetic E2E: `run_ricto.py --synthetic`
- legacy / modern 실데이터는 경로가 있을 때만
- OpenSim SO/JR 연계: ExtLoad MOT 생성 후 기존 파이프라인에서 `APP2_preRiCTO` / `APP2_postRiCTO` 또는 `preRiCTO` / `postRiCTO` 실행

In [ ]:
import os
import sys
import subprocess

repo_root = os.getcwd()
if not os.path.isdir(os.path.join(repo_root, "Codes")):
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

opt_dir = os.path.join(repo_root, "Codes", "c_Run_Tools", "optimization")
os.chdir(opt_dir)
print("cwd:", os.getcwd())


def run_cmd(cmd):
    print("[RUN]", " ".join(cmd), flush=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONPATH"] = os.pathsep.join(
        [os.path.join(repo_root, "Codes"), os.path.join(repo_root, "Codes", "c_Run_Tools"), env.get("PYTHONPATH", "")]
    )
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"exit {rc}")

In [ ]:
# 1) 부호 더미 + 솔버 비교 (데이터 불필요)
run_cmd([sys.executable, "validate_ricto.py", "--dummy", "--solvers", "--box-mass", "7"])

In [ ]:
# 2) synthetic pre/post ExtLoad → Analysis/RiCTO/_synthetic/
run_cmd([
    sys.executable, "run_ricto.py",
    "--synthetic", "--box-mass", "7",
    "--modes", "pre,post",
    "--solver", "least_squares",
])

In [ ]:
# 3) (옵션) legacy OneCycle — OpenSim 데이터가 있을 때만
# run_cmd([sys.executable, "run_ricto.py", "--legacy", "--task", "1", "--box-mass", "15"])

In [ ]:
# 4) (옵션) modern OpenSim_Process 세그먼트
# run_cmd([
#     sys.executable, "run_ricto.py",
#     "--namecode", "SUB2", "--protocol", "Asymmetric",
#     "--condition", "7kg_10bpm", "--segments", "1AB",
#     "--box-mass", "7",
# ])

## OpenSim 연계

1. `run_ricto.py` 가 `ExtLoad_*preRiCTO*` / `*postRiCTO*` (또는 legacy `_estimated_original` / `_RiCTO-corrected`) MOT 작성
2. 레거시: `c_Run Tools/OpenSim_Pipeline.py` 에서 APP=`APP2_preRiCTO` / `APP2_postRiCTO`
3. 현대: `run_opensim_pipeline.py --tools extload,so,jr --apps preRiCTO,postRiCTO`

모델은 베이스 `Scaled.osim` (HeavyHand 질량 모델 아님). RiCTO 분기는 ExtLoad 단.